# Day 48: Implementing Timeouts and Retry Logic in Multi-Agent Graphs

Welcome to Day 48! As we move deeper into Phase 4 (Agent Orchestration & Multi-Agent Systems), we must confront a harsh reality of production AI engineering: **APIs fail, rate limits are hit, and networks latency spikes.**

If your agent graph doesn't gracefully handle rate limits or API timeouts, it will crash entirely or hang indefinitely. Today, we're focusing on adding robust **timeout** and **retry** mechanisms directly into our LangGraph workflows.

## Core Theory: The "Why" and "How"

### Why do we need Timeouts and Retries?
1. **Rate Limits (HTTP 429):** LLM providers (OpenAI, Groq, Anthropic) impose strict rate limits (Requests Per Minute/Tokens Per Minute). In a multi-agent system, multiple agents might hit the API simultaneously, easily triggering rate limits.
2. **Timeouts:** Sometimes the API simply hangs. If your node waits indefinitely, the entire graph execution stalls. We need a hard stop to prevent zombie processes.
3. **Transient Errors (HTTP 500/503):** Temporary server-side glitches that usually resolve if you just try again a few seconds later.

### How do we implement this in LangGraph?
Instead of wrapping every single LLM call in custom `try/except` and `time.sleep()` loops, we can leverage two powerful patterns:
1. **LangChain's Built-in Retries:** LangChain standard runnables (which LangGraph nodes often use) support `with_retry()` out of the box.
2. **Custom Retry Nodes with State:** For more complex logic, we can catch exceptions in a LangGraph node, increment a `retry_count` in our State, and conditionally loop back to the node if the limit hasn't been reached.

In [1]:
import os
from typing import TypedDict, Annotated, Literal
import operator
from langchain_core.messages import HumanMessage, BaseMessage
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
import time
from langchain_core.runnables import RunnableConfig

# Let's set up a dummy key to bypass authentication errors during notebook execution.
# In production, you'd use your real Groq API key.
os.environ["GROQ_API_KEY"] = os.environ.get("GROQ_API_KEY", "gsk_dummy_key")

print("Dependencies loaded successfully.")

Dependencies loaded successfully.


## Implementation 1: The LangChain Native Approach (`with_retry` and `with_config`)

The simplest way to handle retries is to use LangChain's native `.with_retry()` method on the LLM object itself. We can also enforce timeouts using `RunnableConfig`.

Here is how you apply it to a standard LLM:

In [2]:
# 1. Initialize the LLM
# We use a real ChatGroq initialization. The Groq API is extremely fast, making it ideal for agents.
base_llm = ChatGroq(
    model="llama3-8b-8192",
    temperature=0,
    # Note: Max retries is 2 by default in the SDK, but we want explicit control.
    max_retries=0 # We disable native SDK retries to handle it at the LangChain level
)

# 2. Bind retries and timeouts
# We configure it to retry 3 times, with exponential backoff, specifically targeting typical rate limit / timeout errors.
resilient_llm = base_llm.with_retry(
    stop_after_attempt=3,
    wait_exponential_jitter=True
)

def resilient_node(state: dict, config: RunnableConfig):
    """
    A graph node that uses the resilient LLM.
    """
    messages = state.get("messages", [])
    try:
        # We enforce a strict timeout of 5 seconds for the entire operation via the config.
        # If the LLM takes longer, it raises a TimeoutError.
        response = resilient_llm.invoke(
            messages, 
            timeout=5.0
        )
        return {"messages": [response]}
    except Exception as e:
        # If we still fail after retries (e.g., authentication error due to our dummy key),
        # we gracefully catch it.
        error_msg = f"API Call Failed after retries or timeout: {str(e)}"
        return {"messages": [HumanMessage(content=error_msg)]}

print("Resilient node defined.")

Resilient node defined.


## Implementation 2: Graph-Level Retry Logic (State-based)

While `.with_retry()` is great for simple LLM calls, sometimes you want the *Graph* to manage the retries. Why?
Because maybe on a retry, you want to use a *fallback model* (e.g., if Llama 3 70B fails, try Llama 3 8B), or maybe you need to rewrite the prompt before trying again.

We implement this by adding a `retry_count` to our Graph State.

In [3]:
class AgentState(TypedDict):
    messages: Annotated[list, operator.add]
    retry_count: int

def flaking_api_node(state: AgentState):
    """
    Simulates an API that fails constantly to demonstrate graph-level retries.
    """
    current_retries = state.get("retry_count", 0)
    
    print(f"[Node] Attempting call... (Attempt {current_retries + 1})")
    
    # Simulate an API error
    try:
        raise ConnectionError("HTTP 429: Rate Limit Exceeded")
    except Exception as e:
        # We append an error message and increment the retry count
        return {
            "messages": [HumanMessage(content=f"Error: {str(e)}")],
            "retry_count": current_retries + 1
        }

def check_retry(state: AgentState) -> Literal["flaking_api_node", "fallback_node"]:
    """
    Conditional edge to route based on retry count.
    """
    if state.get("retry_count", 0) < 3:
        print("[Router] Retrying...")
        return "flaking_api_node"
    else:
        print("[Router] Max retries reached. Moving to fallback.")
        return "fallback_node"

def fallback_node(state: AgentState):
    """
    A node that executes if the main API fails completely.
    """
    return {
        "messages": [HumanMessage(content="Fallback executed. Could not reach primary API.")]
    }

# Build the graph
builder = StateGraph(AgentState)
builder.add_node("flaking_api_node", flaking_api_node)
builder.add_node("fallback_node", fallback_node)

builder.add_edge(START, "flaking_api_node")
builder.add_conditional_edges(
    "flaking_api_node",
    check_retry
)
builder.add_edge("fallback_node", END)

retry_graph = builder.compile()

print("Graph compiled successfully.")

# Let's test it!
initial_state = {"messages": [HumanMessage(content="Tell me a joke.")], "retry_count": 0}
final_state = retry_graph.invoke(initial_state)

print("\n--- Final Output ---")
print(final_state["messages"][-1].content)

Graph compiled successfully.
[Node] Attempting call... (Attempt 1)
[Router] Retrying...
[Node] Attempting call... (Attempt 2)
[Router] Retrying...
[Node] Attempting call... (Attempt 3)
[Router] Max retries reached. Moving to fallback.

--- Final Output ---
Fallback executed. Could not reach primary API.


## Common Pitfalls in Production
1. **Infinite Loops:** If you implement graph-level retries but forget to increment the `retry_count` in the state (or your reducer logic resets it), your graph will loop infinitely until it hits LangGraph's default recursion limit.
2. **Retrying Non-Transient Errors:** You should not retry HTTP 400 (Bad Request) or 401 (Unauthorized). If the prompt is invalid or the key is wrong, retrying 5 times won't fix it. Only retry 429s (Rate Limits) and 5xxs (Server Errors). You can configure this by catching specific exceptions rather than broad `Exception`s.
3. **Synchronous Sleeping:** If you use `time.sleep()` inside an asynchronous node, you block the entire event loop. If your graph is async, always use `asyncio.sleep()`.

## Practical Lab / Homework
**Task:** Build a robust LangGraph workflow that queries an LLM. Implement a simulated "API failure" mechanism (e.g., use a random number generator to raise an exception 50% of the time). 
Use **Graph-Level State Retries** (like Implementation 2) to attempt the call up to 3 times before giving up and returning a graceful error message to the user.

In [4]:
import random

class LabState(TypedDict):
    messages: Annotated[list, operator.add]
    attempts: int

def unstable_llm_node(state: LabState):
    """
    Simulates an LLM call that randomly fails 50% of the time.
    """
    attempts = state.get("attempts", 0)
    print(f"\n[Unstable Node] Attempting call (Attempt {attempts + 1})...")
    
    # --- YOUR CODE HERE: LAB IMPLEMENTATION ---
    # 1. Generate a random number.
    # 2. If it's < 0.5, print a success message and return a dummy successful response.
    # 3. If it's >= 0.5, raise an Exception.
    # 4. Wrap this in a try/except block to catch the exception, increment attempts, and return the state.
    
    try:
        if random.random() >= 0.5:
            raise ConnectionError("Simulated API Timeout!")
        
        print("[Unstable Node] Success!")
        return {
            "messages": [HumanMessage(content="Successfully generated response!")],
            "attempts": attempts + 1
        }
    except Exception as e:
        print(f"[Unstable Node] Failed: {e}")
        return {
            "messages": [], # Don't add a message yet, wait for fallback or success
            "attempts": attempts + 1
        }

def route_attempts(state: LabState) -> Literal["unstable_llm_node", "failure_node", "__end__"]:
    """
    Router to check if we succeeded or if we need to retry/fail.
    """
    # --- YOUR CODE HERE ---
    # Route logic based on whether we succeeded, need to retry, or failed max times.
    messages = state.get("messages", [])
    attempts = state.get("attempts", 0)
    
    if messages and messages[-1].content == "Successfully generated response!":
        return END
        
    if attempts < 3:
        return "unstable_llm_node"
    return "failure_node"

def failure_node(state: LabState):
    return {
        "messages": [HumanMessage(content="Sorry, the service is currently unavailable after multiple attempts.")]
    }

lab_builder = StateGraph(LabState)
lab_builder.add_node("unstable_llm_node", unstable_llm_node)
lab_builder.add_node("failure_node", failure_node)

lab_builder.add_edge(START, "unstable_llm_node")
lab_builder.add_conditional_edges("unstable_llm_node", route_attempts)
lab_builder.add_edge("failure_node", END)

lab_graph = lab_builder.compile()

print("\n--- Running Lab Test ---")
res = lab_graph.invoke({"messages": [], "attempts": 0})
print(f"\nFinal message: {res['messages'][-1].content}")


--- Running Lab Test ---

[Unstable Node] Attempting call (Attempt 1)...
[Unstable Node] Failed: Simulated API Timeout!

[Unstable Node] Attempting call (Attempt 2)...
[Unstable Node] Failed: Simulated API Timeout!

[Unstable Node] Attempting call (Attempt 3)...
[Unstable Node] Failed: Simulated API Timeout!

Final message: Sorry, the service is currently unavailable after multiple attempts.
